In [ ]:
!apt-get install gmsh
!pip install --upgrade gmsh
!wget "https://fem-on-colab.github.io/releases/fenics-install-real.sh" -O "/tmp/fenics-install.sh" && bash "/tmp/fenics-install.sh"
!pip install meshio

# Importiamo la roccia del modello Blender in gmsh

Import necessari

In [2]:
import gmsh
import json
import meshio
import numpy as np
import pandas as pd
import fenics as fe

from copy import copy

## Import del file json ottenuto da Blender

In [3]:
# Opening JSON file
f = open('Rock_triangoli_normali.json')

# returns JSON object as a dictionary
data = json.load(f)

# Chiusura del file
f.close()

print(len(data))
print("Chiavi:", data[0].keys())

1
Chiavi: dict_keys(['vertices', 'faces', 'normals'])


In [4]:
print("Vertici totali:", len(data[0]["vertices"]))
print("Facce totali:", len(data[0]["faces"]))
print("Normali:", len(data[0]["normals"]))

Vertici totali: 98
Facce totali: 192
Normali: 192


## Import dei punti di training e di testing da forzare nella mesh

## Generazione modello

Inizializziamo il modello e facciamo la mesh su una sola faccia

In [5]:
gmsh.initialize()

starting_model = "1_Base_Roccia"
gmsh.model.add(starting_model)

Integriamo i gmsh tutti i punti contenuti in `data[0]["vertices"]`.

Questi saranno i punti, ovvero gli elementi di dimensione 0 in gmsh. Come `lc` usiamo 1e-14

In [6]:
# Gli inidici dei punti saranno traslati di 1 rispetto agli indici della lista
# data[0]["vertices"]
lc = 1e-14
for el in data[0]["vertices"]:
    ptx = copy(el)
    gmsh.model.geo.addPoint(ptx[0], ptx[1], ptx[2], lc)

Ora aggiungiamo le curve sulla base della list `data[0]["faces]`

In [7]:
# lista di liste con integrate le curve
# la lista più interna permetterà la creazione di ogni superficie
# quella più esterna la creazione automatica di tutte tramite for

lst_cicli = list()

for el in data[0]["faces"]:
    fc = copy(el)
    lst_cicli.append(
        [
            gmsh.model.geo.addLine(fc[0] + 1, fc[1] + 1),
            gmsh.model.geo.addLine(fc[1] + 1, fc[2] + 1),
            gmsh.model.geo.addLine(fc[2] + 1, fc[0] + 1)
        ]
    )

Ora creaiamo le superfici tramite la lista creata precedentemente

In [8]:
lst_surf = list()

for el in lst_cicli:
    el_loop = gmsh.model.geo.addCurveLoop(copy(el))
    lst_surf.append(
        gmsh.model.geo.addPlaneSurface([el_loop])
    )

Aggiungiamo il volume

In [9]:
lst_vol = gmsh.model.geo.addSurfaceLoop(copy(lst_surf))

vol = gmsh.model.geo.addVolume([lst_vol])

Aggiungiamo il gruppo fisico

In [10]:
dim = 3
phy = gmsh.model.addPhysicalGroup(dim, [vol], name="Rock")

In [11]:
gmsh.model.geo.synchronize()

Creiamo la mesh 3D e salviamola

In [12]:
# Creazione della mesh 3D
gmsh.option.setNumber("Mesh.CharacteristicLengthMax", .1)
gmsh.model.mesh.generate(dim)

# Salvataggio
gmsh.write(starting_model + ".msh")

In [13]:
gmsh.finalize()

## Salvataggio in file .xdmf

In [14]:
# Percorso del file .msh
file_path = starting_model + ".msh"

# Carica la mesh utilizzando meshio
mesh  = meshio.read(file_path)

# Estrai i nodi e i connettività degli elementi
points = mesh.points
cells = {"tetra": mesh.cells[0].data}  # Assicurati che la chiave corrisponda al tipo di elemento nel tuo file .msh

# Crea una mesh di FEniCS
mesh_fenics = fe.Mesh()
editor = fe.MeshEditor()
editor.open(mesh_fenics, 'tetrahedron', 3, 3)
editor.init_vertices(len(points))
editor.init_cells(len(cells['tetra']))
for i, point in enumerate(points):
    editor.add_vertex(i, point)
for i, cell in enumerate(cells['tetra']):
    editor.add_cell(i, cell)
editor.close()

# Salva la mesh in formato XDMF
with fe.XDMFFile(mesh_fenics.mpi_comm(), starting_model + ".xdmf") as xdmf:
    xdmf.write(mesh_fenics)